# Práctica voluntariado

## 1. Leer y unir

In [1]:
import json
import unicodedata
from pathlib import Path

import pandas as pd

RAW = Path("../datos/raw")
print(sorted(p.name for p in RAW.iterdir()))

['inscricions.csv', 'quendas.csv']


In [12]:
quendas = pd.read_csv(RAW / "quendas.csv", sep=";", decimal=",")
inscricions = pd.read_csv(RAW / "inscricions.csv")

for nome, df in [("quendas", quendas), ("inscricions", inscricions)]:
    print(f"{nome:11s} {df.shape}  columnas: {list(df.columns)}")

quendas     (704, 5)  columnas: ['data', 'quenda', 'tarefa', 'email', 'horas']
inscricions (140, 5)  columnas: ['nome', 'email', 'data_nacemento', 'concello', 'data_alta']


In [ ]:
quendas.head()

,data,quenda,tarefa,email,horas
0,02/10/2026,mañá,Reparto,elena.souto@correo.example,3.5
1,06/09/2026,mañá,Clasificación,roi.cotelo@correo.example,2.5
2,25/10/2026,mañá,Clasificación,iria.varela@correo.example,2.5
3,30/10/2026,mañá,Clasificación,elena.rial@correo.example,3.0
4,02/09/2026,mañá,Recollida en supermercados,xurxo.fernandez98@correo.example,4.0


In [6]:
quendas.info()

<class 'pandas.DataFrame'>
RangeIndex: 704 entries, 0 to 703
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   data    704 non-null    str    
 1   quenda  704 non-null    str    
 2   tarefa  704 non-null    str    
 3   email   704 non-null    str    
 4   horas   704 non-null    float64
dtypes: float64(1), str(4)
memory usage: 27.6 KB


In [ ]:
inscricions.head()

,nome,email,data_nacemento,concello,data_alta
0,Tareixa Pena,tareixa.pena@correo.example,2006-11-20,A Coruña,2026-01-10
1,Rosa Calvo,rosa.calvo@correo.example,2002-11-01,A Coruña,2026-04-12
2,Martiño Riveiro,martino.riveiro@correo.example,1986-10-10,A Coruña,2026-01-15
3,Roi Lema,roi.lema@correo.example,1982-10-08,A Coruña,2024-12-04
4,Uxío Rial,uxio.rial@correo.example,1973-07-07,Culleredo,2026-01-15


In [7]:
inscricions.info()

<class 'pandas.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   nome            140 non-null    str  
 1   email           140 non-null    str  
 2   data_nacemento  140 non-null    str  
 3   concello        140 non-null    str  
 4   data_alta       140 non-null    str  
dtypes: str(5)
memory usage: 5.6 KB


In [14]:
# quendas["data"] = pd.to_datetime(quendas["data"], format="%d/%m/%Y")
# inscricions["data_alta"] = pd.to_datetime(inscricions["data_alta"], format="%Y-%m-%d")
# inscricions["data_nacemento"] = pd.to_datetime(inscricions["data_nacemento"], format="%Y-%m-%d")

#nomaliez hay 3 errores - hya varios que no cruzan, coreo en vez de correo , algo de puntos
#Convertir fechas con comentarios sobre el funcionamiento del código

def converter_datas(serie, formatos):
    """Proba cada formato por orde e devolve as datas e os textos que non encaixaron en ningún."""
    resultado = pd.Series(pd.NaT, index=serie.index, dtype="datetime64[ns]") #serie vacía para guardar los resultados
    for formato in formatos: #intentamos convertir cada una de las fechas a formato datetime con cada uno de los formatos indicados
        intento = pd.to_datetime(serie, format=formato, errors="coerce") #si un valor no encaja con el formato, lo convierte en NaT
        resultado = resultado.fillna(intento) #guarda el intento que se ha podido convertir
    return resultado, serie[resultado.isna()]

quendas["data"], fallos = converter_datas(quendas["data"], ["%Y-%m-%d", "%d/%m/%Y", "%y%m%d"])
inscricions["data_alta"], fallos = converter_datas(inscricions["data_alta"], ["%Y-%m-%d", "%d/%m/%Y", "%y%m%d"])
inscricions["data_nacemento"], fallos = converter_datas(inscricions["data_nacemento"], ["%Y-%m-%d", "%d/%m/%Y", "%y%m%d"])

print("Sin convertir:", fallos.tolist())#imprimimos intentos que no se han podido convertir con ninguno de los formatos para revisar

Sin convertir: []


In [16]:
inscricions.info()

<class 'pandas.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   nome            140 non-null    str           
 1   email           140 non-null    str           
 2   data_nacemento  140 non-null    datetime64[ns]
 3   concello        140 non-null    str           
 4   data_alta       140 non-null    datetime64[ns]
dtypes: datetime64[ns](2), str(3)
memory usage: 5.6 KB


In [17]:
quendas.info()

<class 'pandas.DataFrame'>
RangeIndex: 704 entries, 0 to 703
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   data    704 non-null    datetime64[ns]
 1   quenda  704 non-null    str           
 2   tarefa  704 non-null    str           
 3   email   704 non-null    str           
 4   horas   704 non-null    float64       
dtypes: datetime64[ns](1), float64(1), str(3)
memory usage: 27.6 KB


In [ ]:
horastotales=quendas["horas"].sum()

np.float64(2090.5)

Normaliza a clave (o correo) nas dúas táboas.

In [20]:
print(sorted(quendas["email"].unique())[:12])

[' alba.cotelo96@correo.example ', ' alba.cotelo@correo.example ', ' alba.rey@correo.example ', ' ana.souto@correo.example ', ' ana.vilar@correo.example ', ' brais.cotelo@correo.example ', ' brais.dopico@correo.example ', ' brais.rey@correo.example ', ' carme.bermudez@correo.example ', ' carme.calvo27@correo.example ', ' carme.cotelo@correo.example ', ' carme.pena@correo.example ']


In [21]:
print(sorted(inscricions["email"].unique())[:12])

['alba.calvo@correo.example', 'alba.castro60@correo.example', 'alba.castro@correo.example', 'alba.cotelo59@correo.example', 'alba.cotelo96@correo.example', 'alba.cotelo@correo.example', 'alba.mourino@correo.example', 'alba.rey@correo.example', 'ana.calvo@correo.example', 'ana.lema@correo.example', 'ana.souto@correo.example', 'ana.vilar@correo.example']


In [ ]:
def normalizar(texto):
    """Minúsculas, sen tiles e sen espazos sobrantes."""
    texto = unicodedata.normalize("NFKD", str(texto).strip().lower())
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return " ".join(texto.split())

inscricions["clave_email"] = inscricions["email"].map(normalizar)
quendas["clave_email"] = quendas["email"].map(normalizar)
print("Nomes distintos nas vendas:", vendas["grupo"].nunique(), "→ claves:", vendas["clave_grupo"].nunique())
